# 02. Analisis exploratorio de las etiquetas

Este cuaderno estudia las etiquetas del conjunto de datos. Analiza la magnitud del dano como
variable cuantitativa, describe las variables categoricas, examina las relaciones entre ellas y
evalua dos riesgos que condicionan el diseno de cualquier modelo posterior: la agrupacion de
fotografias por campo y el desplazamiento de las distribuciones entre temporadas. Parte de los
metadatos guardados por el cuaderno `01_adquisicion_datos` y no vuelve a leer los CSV originales.

## Entorno

In [ ]:
import sys
from pathlib import Path


def raiz_proyecto():
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "src").is_dir():
            return candidato
    raise RuntimeError("No se encontro la raiz del proyecto")


RAIZ = raiz_proyecto()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

%matplotlib inline
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

from src import carga, graficos, limpieza, tablas
from src.config import ORDEN_DANOS, ORDEN_ETAPAS, TEMPORADAS, TIPOS_DANO

graficos.aplicar_estilo()

train = carga.cargar_metadatos("train")
test = carga.cargar_metadatos("test")
print(f"train: {train.shape[0]} registros, {train.shape[1]} columnas")
print(f"test:  {test.shape[0]} registros, {test.shape[1]} columnas")

## Estructura del conjunto

El diccionario de datos distingue las variables originales de la competencia (identificador,
archivo, etapa, dano, magnitud, temporada) de las derivadas durante la preparacion (productor,
campo, cultivo, sitio, tipo de captura, indicadores de calidad). El conteo por escala de medicion
resume que tipo de variable predomina.

In [ ]:
diccionario = carga.diccionario_datos(train)[["variable", "tipo", "escala", "nulos", "unicos"]]
diccionario

In [ ]:
diccionario["escala"].value_counts()

## Variable objetivo: magnitud del dano

La magnitud es un porcentaje de perdida discreto, acotado entre cero y cien y registrado en pasos
de diez. Se describe su distribucion, su fuerte concentracion en cero, y por que el criterio
clasico de valores atipicos no es apropiado para esta variable.

In [ ]:
tablas.resumen_numerico(train, ["magnitud"])

In [ ]:
print(f"proporcion con magnitud 0:   {(train['magnitud'] == 0).mean():.4f}")
print(f"proporcion con magnitud 100: {(train['magnitud'] == 100).mean():.4f}")

In [ ]:
tablas.tabla_frecuencia(train, "magnitud")

In [ ]:
_ = graficos.histograma(
    train["magnitud"], "Distribucion de la magnitud", "magnitud", nombre_archivo="02_histograma_magnitud"
)

In [ ]:
_ = graficos.ecdf(
    train["magnitud"], "Distribucion acumulada de la magnitud", "magnitud", nombre_archivo="02_ecdf_magnitud"
)

In [ ]:
tablas.resumen_atipicos(train, ["magnitud"])

El criterio del rango intercuartilico no es apropiado aqui. Como la magnitud esta fuertemente
concentrada en cero, el primer y el tercer cuartil valen cero, el rango intercuartilico es nulo y
el criterio marca como atipico cualquier valor positivo. Pero los valores positivos son
precisamente el fenomeno de interes, de modo que aplicar esa regla contradiria el proposito del
analisis. No se eliminan esos registros.

## La magnitud como variable condicionada al diagnostico de sequia

La regla estructural y el resumen de la magnitud por tipo de dano muestran que la magnitud toma
valores positivos casi exclusivamente cuando el dano declarado es sequia. Esto es una regla de
construccion del conjunto, no una correlacion estadistica, y separa dos preguntas distintas: si la
fotografia corresponde a un caso de sequia, y en caso afirmativo, que tan severa es.

In [ ]:
limpieza.regla_estructural(train)

In [ ]:
resumen_por_dano = train.groupby("dano", observed=True)["magnitud"].agg(
    tamano="size", media="mean", maximo="max"
)
resumen_por_dano["descripcion"] = resumen_por_dano.index.map(TIPOS_DANO)
resumen_por_dano

La magnitud media es practicamente cero en todos los tipos de dano salvo sequia, y solo la sequia
alcanza magnitudes altas. El problema util no es estimar la magnitud sobre el conjunto completo,
sino sobre el subconjunto diagnosticado con sequia.

## Consecuencia sobre el conjunto de prueba

El archivo de prueba distribuye el tipo de dano junto con las imagenes. Si la regla anterior se
sostiene, los registros de prueba con un tipo de dano distinto de sequia tienen una respuesta de
magnitud conocida de antemano, igual a cero, y solo los registros de sequia exigen estimar un
valor.

In [ ]:
no_sequia_test = int((test["dano"] != "DR").sum())
sequia_test = int((test["dano"] == "DR").sum())
print(f"registros de prueba con dano distinto de sequia (magnitud 0 conocida): {no_sequia_test}")
print(f"registros de prueba con sequia (exigen estimar): {sequia_test}")
print(f"proporcion determinada de antemano: {no_sequia_test / len(test):.4f}")

In [ ]:
tablas.comparar_distribuciones_categoricas(train, test, "dano")

Una porcion considerable del conjunto de prueba queda determinada sin mirar la imagen. El
planteamiento util del problema es estimar la magnitud sobre el subconjunto diagnosticado con
sequia y no sobre el conjunto completo.

## Distribucion de la magnitud dentro del subconjunto de sequia

Al aislar las filas cuyo dano es sequia, la magnitud deja de estar dominada por el cero y su forma
cambia. El diagrama de caja por temporada revela si la severidad tipica difiere entre temporadas,
que es la evidencia mas directa del fenomeno de perdida de desempeno que motiva el desafio.

In [ ]:
sequia = train[train["dano"] == "DR"].copy()
print(f"registros de sequia en entrenamiento: {len(sequia)}")
tablas.resumen_numerico(sequia, ["magnitud"])

In [ ]:
tablas.tabla_frecuencia(sequia, "magnitud")

In [ ]:
_ = graficos.histograma(
    sequia["magnitud"], "Magnitud dentro del subconjunto de sequia", "magnitud",
    nombre_archivo="02_histograma_magnitud_sequia",
)

In [ ]:
_ = graficos.caja_por_categoria(
    sequia, "temporada", "magnitud", "Magnitud de sequia por temporada",
    nombre_archivo="02_caja_magnitud_sequia_temporada",
)

In [ ]:
tablas.resumen_por_grupo(sequia, "temporada", "magnitud")

La severidad tipica de la sequia cambia de forma marcada entre temporadas. Un modelo que aprende
la relacion entre la imagen y la magnitud en unas temporadas encuentra en otra una distribucion de
severidad distinta, lo que explica en parte la caida de desempeno entre temporadas.

## Variables categoricas

Se describen las tres variables categoricas centrales: tipo de dano, etapa de crecimiento y
temporada. Las temporadas se nombran segun el regimen de lluvias de Africa oriental: LR para
lluvias largas y SR para lluvias cortas, seguidas del ano.

In [ ]:
frecuencia_dano = tablas.tabla_frecuencia(train, "dano")
frecuencia_dano["descripcion"] = frecuencia_dano["dano"].map(TIPOS_DANO)
frecuencia_dano

In [ ]:
display(tablas.tabla_frecuencia(train, "etapa"))
display(tablas.tabla_frecuencia(train, "temporada"))

In [ ]:
_ = graficos.barras_frecuencia(
    tablas.tabla_frecuencia(train, "dano"), "Frecuencia por tipo de dano", nombre_archivo="02_barras_dano"
)

In [ ]:
_ = graficos.barras_frecuencia(
    tablas.tabla_frecuencia(train, "etapa"), "Frecuencia por etapa", nombre_archivo="02_barras_etapa"
)

In [ ]:
_ = graficos.barras_frecuencia(
    tablas.tabla_frecuencia(train, "temporada"), "Frecuencia por temporada", nombre_archivo="02_barras_temporada"
)

El crecimiento sano es la categoria mas frecuente y la sequia concentra la mayoria de los casos
con dano, coherente con que la sequia sea el motivo dominante de reclamacion.

## Tipo de captura

Cada tipo de captura tiene un proposito: la inicial es una referencia al comenzar el ciclo, el
seguimiento registra la evolucion, y el reclamo acompana un reporte de dano. Como cada uno tiene
una magnitud media distinta, un cambio en la composicion del tipo de captura entre temporadas
altera la distribucion de la magnitud aunque el terreno no cambie.

In [ ]:
display(tablas.tabla_frecuencia(train, "tipo_captura"))
tablas.resumen_por_grupo(train, "tipo_captura", "magnitud")

In [ ]:
display(tablas.tabla_contingencia(train, "temporada", "tipo_captura"))
tablas.tabla_contingencia(train, "temporada", "tipo_captura", normalizar="fila")

In [ ]:
_ = graficos.barras_apiladas(
    tablas.tabla_contingencia(train, "temporada", "tipo_captura", normalizar="fila"),
    "Composicion del tipo de captura por temporada", "proporcion",
    nombre_archivo="02_apiladas_temporada_captura",
)

La composicion del tipo de captura no es homogenea entre temporadas. Como el reclamo concentra las
magnitudes altas, las temporadas con mayor proporcion de reclamos tienden a mostrar magnitudes
mas severas, un efecto de composicion que se suma a cualquier diferencia agronomica real.

## Cruce entre variables

Se cruzan la magnitud con el tipo de dano y con la etapa, y la etapa con el tipo de dano, para
entender el mecanismo que conecta la etapa del cultivo con la aparicion del dano.

In [ ]:
display(tablas.resumen_por_grupo(train, "dano", "magnitud"))
_ = graficos.caja_por_categoria(
    train, "dano", "magnitud", "Magnitud por tipo de dano", nombre_archivo="02_caja_magnitud_dano"
)

In [ ]:
display(tablas.resumen_por_grupo(train, "etapa", "magnitud"))
_ = graficos.caja_por_categoria(
    train, "etapa", "magnitud", "Magnitud por etapa", nombre_archivo="02_caja_magnitud_etapa"
)

In [ ]:
display(tablas.tabla_contingencia(train, "etapa", "dano"))
tablas.tabla_contingencia(train, "etapa", "dano", normalizar="fila")

In [ ]:
_ = graficos.barras_apiladas(
    tablas.tabla_contingencia(train, "etapa", "dano", normalizar="fila"),
    "Composicion del tipo de dano por etapa", "proporcion",
    nombre_archivo="02_apiladas_etapa_dano",
)

El dano por sequia se concentra en las etapas tardias del ciclo. Esto sugiere que la etapa no
actua como causa independiente sino como indicador del momento en que el dano se vuelve
observable en la fotografia.

## Composicion por temporada

Se examina como cambian entre temporadas la composicion del tipo de dano, la magnitud y la
composicion de la etapa.

In [ ]:
_ = graficos.barras_apiladas(
    tablas.tabla_contingencia(train, "temporada", "dano", normalizar="fila"),
    "Composicion del tipo de dano por temporada", "proporcion",
    nombre_archivo="02_apiladas_temporada_dano",
)

In [ ]:
display(tablas.resumen_por_grupo(train, "temporada", "magnitud"))
_ = graficos.caja_por_categoria(
    train, "temporada", "magnitud", "Magnitud por temporada", nombre_archivo="02_caja_magnitud_temporada"
)

In [ ]:
_ = graficos.barras_apiladas(
    tablas.tabla_contingencia(train, "temporada", "etapa", normalizar="fila"),
    "Composicion de la etapa por temporada", "proporcion",
    nombre_archivo="02_apiladas_temporada_etapa",
)

## Medidas de asociacion

La correlacion de Pearson no es adecuada porque las variables explicativas son categoricas y la
magnitud es discreta y acotada. Se usa la V de Cramer entre pares de variables categoricas y el
eta cuadrado para medir cuanta varianza de la magnitud explica cada variable categorica.

In [ ]:
variables_categoricas = ["dano", "etapa", "temporada", "tipo_captura"]
_ = graficos.mapa_calor(
    tablas.matriz_v_cramer(train, variables_categoricas),
    "Asociacion entre variables categoricas (V de Cramer)", nombre_archivo="02_mapa_cramers",
)

In [ ]:
tablas.tabla_eta_cuadrado(train, variables_categoricas, "magnitud")

El tipo de dano explica la mayor parte de la varianza de la magnitud, coherente con que la
magnitud este condicionada al diagnostico de sequia. La temporada explica poca varianza sobre el
conjunto completo, pero esa lectura es incompleta: como se vio en la seccion de sequia, el efecto
de la temporada se concentra dentro del subconjunto diagnosticado con sequia, no sobre el
conjunto entero.

## Ruido de etiquetado

Se inspeccionan directamente las filas que rompen la regla estructural, para que el lector pueda
examinar los casos concretos en vez de solo un conteo agregado.

In [ ]:
limpieza.regla_estructural(train)

In [ ]:
columnas_inspeccion = ["id", "temporada", "etapa", "dano", "magnitud", "tipo_captura"]
sequia_sin_magnitud = train[(train["dano"] == "DR") & (train["magnitud"] == 0)]
print(f"sequia declarada con magnitud cero: {len(sequia_sin_magnitud)} registros")
sequia_sin_magnitud[columnas_inspeccion].head(15)

## Agrupacion de fotografias por campo

Las fotografias de un mismo campo comparten terreno, cultivo y con frecuencia dispositivo de
captura, de modo que son observaciones dependientes. Una particion aleatoria que reparta las fotos
de un campo entre entrenamiento y validacion permitiria a un modelo reconocer el campo en lugar
del dano. Se cuantifica la agrupacion y se simula una particion aleatoria para medir esa fuga.

In [ ]:
fotos_por_campo = train.groupby("id_campo").size()
print(f"campos distintos: {fotos_por_campo.shape[0]}")
print(f"mediana de fotos por campo: {fotos_por_campo.median():.1f}")
print(f"maximo de fotos por campo: {fotos_por_campo.max()}")
print(f"proporcion de campos con mas de una foto: {(fotos_por_campo > 1).mean():.4f}")

In [ ]:
tope = fotos_por_campo.quantile(0.99)
_ = graficos.histograma(
    fotos_por_campo[fotos_por_campo <= tope], "Fotografias por campo (recorte al percentil 99)",
    "fotografias por campo", bins=30, nombre_archivo="02_histograma_fotos_por_campo",
)

In [ ]:
rng = np.random.default_rng(42)
mezcla = rng.permutation(train.index.to_numpy())
corte = int(0.8 * len(mezcla))
indices_entrenamiento, indices_validacion = mezcla[:corte], mezcla[corte:]

campos_entrenamiento = set(train.loc[indices_entrenamiento, "id_campo"].dropna())
campos_validacion = set(train.loc[indices_validacion, "id_campo"].dropna())
campos_compartidos = campos_entrenamiento & campos_validacion
registros_afectados = train.loc[indices_validacion, "id_campo"].isin(campos_compartidos).mean()

print(f"campos presentes en ambos lados de la particion aleatoria: {len(campos_compartidos)}")
print(f"proporcion de registros de validacion en campos compartidos: {registros_afectados:.4f}")

Una particion aleatoria simple deja una fraccion alta de los registros de validacion en campos que
tambien aparecen en entrenamiento. Cualquier validacion honesta debe separar por campo, no al azar.

## Solapamiento de campos entre entrenamiento y prueba

Se verifica como esta construida la particion oficial de la competencia: cuantos campos y
productores aparecen en ambos conjuntos, y que proporcion de los registros de cada conjunto cae en
campos compartidos.

In [ ]:
campos_train = set(train["id_campo"].dropna())
campos_test = set(test["id_campo"].dropna())
campos_comunes = campos_train & campos_test
productores_comunes = set(train["id_productor"].dropna()) & set(test["id_productor"].dropna())

print(f"campos en entrenamiento: {len(campos_train)}")
print(f"campos en prueba: {len(campos_test)}")
print(f"campos presentes en ambos: {len(campos_comunes)}")
print(f"productores presentes en ambos: {len(productores_comunes)}")
print(f"proporcion de registros de entrenamiento en campos compartidos: {train['id_campo'].isin(campos_comunes).mean():.4f}")
print(f"proporcion de registros de prueba en campos compartidos: {test['id_campo'].isin(campos_comunes).mean():.4f}")

La particion oficial no separa los campos entre entrenamiento y prueba: 2414 de los 2475 campos de
prueba tambien aparecen en entrenamiento, y alrededor del 80 por ciento de los registros de
entrenamiento y del 88 por ciento de los de prueba caen en campos compartidos. Un modelo puede
aprender a reconocer el campo en lugar del dano, de modo que el desempeno reportado en una tabla de
posiciones basada en esta particion esta inflado por fuga de informacion y no mide generalizacion
real.

## Desplazamiento entre entrenamiento y prueba

Se comparan las distribuciones de temporada, etapa y tipo de dano entre entrenamiento y prueba con
la distancia de variacion total, y se grafica la composicion por temporada.

In [ ]:
for variable in ("temporada", "etapa", "dano"):
    distancia = tablas.distancia_variacion_total(train, test, variable)
    print(f"distancia de variacion total en {variable}: {distancia:.4f}")

In [ ]:
composicion_temporada = tablas.comparar_distribuciones_categoricas(train, test, "temporada").set_index("temporada")
display(composicion_temporada)
_ = graficos.barras_comparativas(
    composicion_temporada, ["train", "test"], "Composicion por temporada: entrenamiento y prueba",
    "proporcion", nombre_archivo="02_comparacion_temporada",
)

Las distribuciones de estas variables son casi identicas entre entrenamiento y prueba, con
distancias de variacion total por debajo de 0.01. Combinado con el fuerte solapamiento de campos
de la seccion anterior, esto indica que la particion oficial se construyo de forma aleatoria y
estratificada, sin separar por campo ni por temporada. Por lo tanto el desplazamiento que hace
fallar a los modelos entre temporadas no es visible al evaluar sobre esta particion oficial: hay
que buscarlo comparando temporadas directamente, tanto en las etiquetas como en las imagenes.

## Cierre de la etapa

El analisis tabular delimita el problema desde el lado de las etiquetas: la magnitud esta
condicionada al diagnostico de sequia, su severidad se desplaza entre temporadas, la particion
oficial no separa por campo ni expone el desplazamiento entre temporadas, y existe ruido de
etiquetado acotado. El cuaderno `03_eda_imagenes` incorpora los atributos visuales de las
fotografias para verificar si las diferencias entre temporadas tambien se manifiestan en las
condiciones de captura.